In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M25.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7678710216947133, 'n_it': 0.39601941095614723}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}
            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
18.24527842152752

Trial 1 =========================================
13.841962745294762

Trial 2 =========================================
13.885200090764659

Trial 3 =========================================
18.259810090883427

Trial 4 =========================================
16.92824469981826

Trial 5 =========================================
18.278637050899388

Trial 6 =========================================
18.22837752750082

Trial 7 =========================================
18.273482572276333

Trial 8 =========================================
18.263025457014248

Trial 9 =========================================
18.09449582340571

Trial 10 =========================================
18.184047017272324

Trial 11 =========================================
18.136441611036705

Trial 12 =========================================
18.294416765548398

Trial 13 =========================================
18.184445453507593

Trial 14 ===========

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 30 =========================================
18.235521713983008

Trial 31 =========================================
18.150125477014925

Trial 32 =========================================
18.28650160878345

Trial 33 =========================================
13.910324359117098

Trial 34 =========================================
18.240507932699742

Trial 35 =========================================
18.260849257099323

Trial 36 =========================================
18.293102250048328

Trial 37 =========================================
14.897864438632402

Trial 38 =========================================
13.908034637026727

Trial 39 =========================================
17.424373345764742

Trial 40 =========================================
13.90672248516598

Trial 41 =========================================
17.78897713911745

Trial 42 =========================================
18.287230078457092

Trial 43 =========================================
13.881650328196942

Trial 44 

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 65 =========================================
17.930805242224004

Trial 66 =========================================
13.883886240149337

Trial 67 =========================================
13.907701586403867

Trial 68 =========================================
18.212304897789707

Trial 69 =========================================
17.47387290895724

Trial 70 =========================================
18.134171765021904

Trial 71 =========================================
18.283673625837963

Trial 72 =========================================
18.263348120445507

Trial 73 =========================================
18.28778035794351

Trial 74 =========================================
13.895981200153997

Trial 75 =========================================
18.125584793845558

Trial 76 =========================================
13.858342364689893

Trial 77 =========================================
18.278967879382392

Trial 78 =========================================
13.890336948459721

Trial 79

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.294416765548398
Avg = 16.709730583221457
Std = 1.94635528467584


In [7]:
print(y_max_arr.tolist())

[18.24527842152752, 13.841962745294762, 13.885200090764659, 18.259810090883427, 16.92824469981826, 18.278637050899388, 18.22837752750082, 18.273482572276333, 18.263025457014248, 18.09449582340571, 18.184047017272324, 18.136441611036705, 18.294416765548398, 18.184445453507593, 18.291873833351953, 17.127050376016022, 18.278563741577187, 16.50077953517469, 18.271620033022902, 18.256490684013595, 18.27297992304927, 13.902738037679619, 18.21967363670862, 13.903266745729505, 13.889136208009312, 18.144125965946206, 15.505956585763954, 13.801024598066421, 18.282688931453336, 18.227880943590804, 18.235521713983008, 18.150125477014925, 18.28650160878345, 13.910324359117098, 18.240507932699742, 18.260849257099323, 18.293102250048328, 14.897864438632402, 13.908034637026727, 17.424373345764742, 13.90672248516598, 17.78897713911745, 18.287230078457092, 13.881650328196942, 17.622293258662193, 18.22969851611173, 13.909042479394525, 13.899462483026024, 18.271903994692305, 18.278703625607776, 18.0436437

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M25/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    18.274787
1    13.770174
2    18.076576
3    18.270455
4    17.932169
..         ...
295  18.117418
296  17.506980
297  18.236165
298  13.908743
299  13.896196

[300 rows x 1 columns]
